In [ ]:
!pip install -q ucimlrepo
!pip install -q optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 7.4 MB/s eta 0:00:00


In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/sequence_prediction_preactice

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/sequence_prediction_preactice


In [ ]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim


from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error as rmse

# Importing tqdm for prgress bar
from tqdm import tqdm

# Importing optun for hyperparameter tuning
import optuna

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Setting up necessary seeds for reproducibility across runs.
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.benchmark=False
torch.use_deterministic_algorithms(True)

In [ ]:
import dagshub
dagshub.init(repo_owner='redowansakib01', repo_name='cora_gnn', mlflow=True)

In [ ]:
import mlflow

# The set_experiment API creates a new experiment if it doesn't exist.
mlflow.set_experiment("PM25_LSTM")
mlflow.set_tracking_uri('https://dagshub.com/redowansakib01/cora_gnn.mlflow')

# IMPORTANT: Enable system metrics monitoring
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)

In [ ]:
  # fetch dataset
beijing_pm2_5 = fetch_ucirepo(id=381)

# data (as pandas dataframes)
X = beijing_pm2_5.data.features
y = beijing_pm2_5.data.targets

# metadata
print(beijing_pm2_5.metadata)

X.index = pd.to_datetime(X[['year', 'month', 'day', 'hour']])
X = X.drop(columns = ['year', 'month', 'day', 'hour'])

y = y.copy()

y.loc[:,'nan_mask'] = (~y['pm2.5'].isnull()).astype(int)
y = y.fillna(0)

{'uci_id': 381, 'name': 'Beijing PM2.5', 'repository_url': 'https://archive.ics.uci.edu/dataset/381/beijing+pm2+5+data', 'data_url': 'https://archive.ics.uci.edu/static/public/381/data.csv', 'abstract': 'This hourly data set contains the PM2.5 data of US Embassy in Beijing. Meanwhile, meteorological data from Beijing Capital International Airport are also included. ', 'area': 'Climate and Environment', 'tasks': ['Regression'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 43824, 'num_features': 11, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['pm2.5'], 'index_col': ['No'], 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2015, 'last_updated': 'Sat Mar 16 2024', 'dataset_doi': '10.24432/C5JS49', 'creators': ['Song Chen'], 'intro_paper': {'ID': 432, 'type': 'NATIVE', 'title': "Assessing Beijing's PM2.5 pollution: severity, weather impact, APEC and winter heating", 'authors': 'Xuan Liang, T. Zou, Bi

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 43824 entries, 2010-01-01 00:00:00 to 2014-12-31 23:00:00
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   DEWP    43824 non-null  int64  
 1   TEMP    43824 non-null  float64
 2   PRES    43824 non-null  float64
 3   cbwd    43824 non-null  object 
 4   Iws     43824 non-null  float64
 5   Is      43824 non-null  int64  
 6   Ir      43824 non-null  int64  
dtypes: float64(3), int64(3), object(1)
memory usage: 2.7+ MB


In [ ]:
X.head()

,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
2010-01-01 00:00:00,-21,-11.0,1021.0,NW,1.79,0,0
2010-01-01 01:00:00,-21,-12.0,1020.0,NW,4.92,0,0
2010-01-01 02:00:00,-21,-11.0,1019.0,NW,6.71,0,0
2010-01-01 03:00:00,-21,-14.0,1019.0,NW,9.84,0,0
2010-01-01 04:00:00,-20,-12.0,1018.0,NW,12.97,0,0


In [ ]:
original_columns = X.columns

In [ ]:
X.cbwd.unique()

array(['NW', 'cv', 'NE', 'SE'], dtype=object)

In [ ]:
wind_mask = (X.cbwd == 'cv').astype(int)

wind_map = {
    'cv' : 0,
    'NE' : 1,
    'NW' : 2,
    'SE' : 3,
    'SW' : 4
}

X.loc[:,'cbwd'] = X['cbwd'].replace(wind_map)

X.loc[:,'wdm'] = wind_mask

In [ ]:
trian_size = int(len(X) * 0.8)
val_size = int(len(X) * 0.1)
test_size = len(X) - trian_size - val_size

train_X, train_y = X.iloc[:trian_size], y.iloc[:trian_size]
val_X, val_y = X.iloc[trian_size:trian_size+val_size], y.iloc[trian_size:trian_size+val_size]
test_X, test_y = X.iloc[trian_size+val_size:], y.iloc[trian_size+val_size:]


In [ ]:
scaler = ColumnTransformer([
    ('ss', StandardScaler(), original_columns)
])

train_X = scaler.fit_transform(train_X)
val_X = scaler.transform(val_X)
test_X = scaler.transform(test_X)

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, lookback, forecast_steps):
        """
        X: A numpy array of shape (n_samples, n_features)
        y: A numpy array of shape (n_samples, 2)
           - y[:, 0]: the target values (e.g., PM2.5, NaNs filled with 0)
           - y[:, 1]: the mask (1 for valid data, 0 for missing data)
        lookback: The size of the historical sliding window (e.g., 24)
        forecast_steps: Number of future timesteps to predict (e.g., 6)
        """
        assert len(X) == len(y), 'Length of X and y must be the same'

        # Store original values as clean tensors
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_vals = torch.tensor(y[:, 0], dtype=torch.float32)
        self.y_mask = torch.tensor(y[:, 1], dtype=torch.float32)

        self.lookback = lookback
        self.forecast_steps = forecast_steps

    def __len__(self):
        # The total number of valid starting positions.
        # We subtract lookback AND forecast_steps to prevent slicing past the end of the array.
        return len(self.X) - self.lookback - self.forecast_steps + 1

    def __getitem__(self, idx):
        # In a contiguous dataset, the passed 'idx' maps perfectly to the window's starting point
        start_idx = idx

        # 1. Slice the historical feature window: shape (lookback, n_features)
        X_window = self.X[start_idx : start_idx + self.lookback]

        # The future target window starts exactly where the feature window ends
        target_start = start_idx + self.lookback
        target_end = target_start + self.forecast_steps

        # 2. Slice the multi-step true target values: shape (forecast_steps,)
        y_target = self.y_vals[target_start : target_end]

        # 3. Slice the corresponding missing data flags: shape (forecast_steps,)
        y_mask_window = self.y_mask[target_start : target_end]

        # Output all three elements simultaneously
        return X_window, y_target, y_mask_window

In [ ]:
lookback = 7*24
forecast_steps = 24

train_dataset = TimeSeriesDataset(train_X.values,
                                  train_y.values,
                                  lookback=lookback,
                                  forecast_steps=forecast_steps)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = TimeSeriesDataset(val_X.values,
                                val_y.values,
                                lookback=lookback,
                                forecast_steps=forecast_steps)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

test_dataset = TimeSeriesDataset(test_X.values,
                                 test_y.values,
                                 lookback=lookback,
                                 forecast_steps=forecast_steps)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
class PM25LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, forecast_steps):
        super(PM25LSTM, self).__init__()

        self.lstm = nn.LSTM(input_size,
                            hidden_size,
                            num_layers,
                            dropout=dropout,
                            batch_first=True)

        self.fc = nn.Linear(hidden_size, forecast_steps)

    def forward(self, x):
        x, _ = self.lstm(x)

        x = x[:, -1, :]

        x = self.fc(x)
        return x

In [ ]:
class TimeSeriesTrainer:
    def __init__(self,
                 model,
                 optimizer,
                 criterion,
                 l1_lambda=0.0,
                 device="cpu",
                 metric=None,
                 metric_kws=None):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.l1_lambda = l1_lambda
        self.device = device
        self.metric = metric
        self.metric_kws = metric_kws if metric_kws is not None else {}

    def train_one_epoch(self, train_loader):
        self.model.train()
        total_loss = 0.0

        all_y_hat = []
        all_y = []
        all_masks = []

        for X, y, y_mask in train_loader:
            # 1. Move ALL tensors to the correct device first
            X = X.to(self.device)
            y = y.to(self.device)
            y_mask = y_mask.to(self.device).to(torch.float32) # Ensure mask is float for multiplication

            self.optimizer.zero_grad()

            # Forward pass keeps clean shape: (batch_size, forecast_steps)
            y_hat = self.model(X)

            # 2. Compute Element-wise Loss (CRUCIAL: reduction='none')
            # This calculates the loss for every point without averaging them yet
            loss_matrix = self.criterion(y_hat, y) # Shape: (batch_size, forecast_steps)

            # 3. Apply the mask! Multiply by 0 where data is missing, 1 where valid
            masked_loss = loss_matrix * y_mask

            # 4. Average only over the VALID elements to prevent bias
            # Avoid division by zero if an entire batch happens to be missing values
            num_valid_elements = torch.sum(y_mask)
            if num_valid_elements > 0:
                loss = torch.sum(masked_loss) / num_valid_elements
            else:
                loss = torch.sum(masked_loss) # fallback

            # Calculate L1 regularization if enabled
            if self.l1_lambda > 0:
                l1_norm = sum(p.abs().sum() for p in self.model.parameters())
                loss = loss + (self.l1_lambda * l1_norm)

            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

            # 5. Keep full 2D arrays intact for consistent stacking
            all_y_hat.append(y_hat.detach().cpu().numpy())
            all_y.append(y.cpu().numpy())
            all_masks.append(y_mask.cpu().numpy())

        avg_loss = total_loss / len(train_loader)

        # 6. Calculate validation metrics ONLY on valid elements
        score = None
        if self.metric and len(all_y) > 0:
            all_y_hat = np.vstack(all_y_hat)
            all_y = np.vstack(all_y)
            all_masks = np.vstack(all_masks).astype(bool) # Convert to boolean for indexing

            # Filter out missing values cleanly right before passing to metrics
            valid_y = all_y[all_masks]
            valid_y_hat = all_y_hat[all_masks]

            score = self.metric(valid_y, valid_y_hat, **self.metric_kws)

        return score, avg_loss

    def validate(self, val_loader):
        self.model.eval()
        total_val_loss = 0.0

        all_preds = []
        all_targets = []
        all_masks = []

        with torch.no_grad():
            for X, y, y_mask in val_loader:
                X = X.to(self.device)
                y = y.to(self.device)
                y_mask = y_mask.to(self.device).to(torch.float32)

                output = self.model(X)

                # Apply masked loss validation paradigm
                loss_matrix = self.criterion(output, y)
                masked_loss = loss_matrix * y_mask

                num_valid_elements = torch.sum(y_mask)
                val_loss = torch.sum(masked_loss) / (num_valid_elements if num_valid_elements > 0 else 1)
                total_val_loss += val_loss.item()

                all_preds.append(output.cpu().numpy())
                all_targets.append(y.cpu().numpy())
                all_masks.append(y_mask.cpu().numpy())

        avg_val_loss = total_val_loss / len(val_loader)

        score = None
        if self.metric and len(all_targets) > 0:
            all_preds = np.vstack(all_preds)
            all_targets = np.vstack(all_targets)
            all_masks = np.vstack(all_masks).astype(bool)

            score = self.metric(all_targets[all_masks], all_preds[all_masks], **self.metric_kws)

        return score, avg_val_loss

In [ ]:
class Objective:
    def __init__(self,
                 train_loader,
                 val_loader,
                 input_size,
                 forecast_steps,
                 device,
                 nepoch,
                 patience,
                 grace):

            """
            Args:
                train_loader: DataLoader for training
                val_loader: DataLoader for validation
                input_size: Integer number of input size (input_size)
                n_classes: Integer number of output classes (out_size)
                forecast_steps: Integer number of forecast steps
                device: torch.device('cuda' or 'cpu')
            """
            self.train_loader = train_loader
            self.val_loader = val_loader
            self.input_size = input_size
            self.forecast_steps = forecast_steps
            self.device = device
            self.nepoch = nepoch
            self.patience = patience
            self.grace = grace

    def __call__(self, trial):
        # 1. Suggest architectural parameters
        num_layers = trial.suggest_int('num_layers', 1, 3)
        hidden_size = trial.suggest_categorical("hidden_size", [16, 32, 64, 128, 256])
        dropout = trial.suggest_float('dropout', 0.0, 0.5)

        # Fix PyTorch nn.LSTM single layer dropout warning/constraint
        lstm_dropout = 0.0 if num_layers == 1 else dropout

        # 2. Safely build model architecture
        try:
            self.model = PM25LSTM(
                input_size=self.input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                dropout=lstm_dropout,
                forecast_steps=self.forecast_steps
            ).to(self.device)
        except (RuntimeError, ValueError) as e:
            print(f"Trial {trial.number} Pruned due to invalid architecture: {e}")
            raise optuna.exceptions.TrialPruned()

        # 3. Suggest Optimization settings
        optim_lr = trial.suggest_float("optim_lr", 1e-4, 5e-2, log=True)
        l1_lambda = trial.suggest_float("l1_lambda", 1e-6, 1e-2, log=True)
        l2_lambda = trial.suggest_float("l2_lambda", 1e-6, 1e-2, log=True)

        criterion = nn.MSELoss()

        # Pass L2 lambda parameter cleanly into weight_decay natively
        optimizer = optim.Adam(self.model.parameters(), lr=optim_lr, weight_decay=l2_lambda)

        # Instantiate our optimized TimeSeriesTrainer class
        trainer = TimeSeriesTrainer(
            model=self.model,
            optimizer=optimizer,
            criterion=criterion,
            l1_lambda=l1_lambda,
            device=self.device,
            metric=rmse # Explicit helper function reference
        )

        pbar = tqdm(range(self.nepoch), desc=f"Trial {trial.number}", leave=True)

        # CORRECT MINIMIZATION TRACKING PARADIGM
        best_rmse = float('inf')  # Start tracking from positive infinity
        patience_counter = 0
        best_epoch = 0

        for iepoch in pbar:
            # Train and log metrics
            train_score, train_loss = trainer.train_one_epoch(self.train_loader)
            # Validate and log metrics
            val_rmse, val_loss = trainer.validate(self.val_loader)

            # Update progress bar metrics
            pbar.set_postfix({"Train Loss": f"{train_loss:.4f}", "Val RMSE": f"{val_rmse:.4f}"})

            # Early Stopping Evaluation (Looking for LOWER error)
            if val_rmse < best_rmse:
                best_rmse = val_rmse
                best_epoch = iepoch
                patience_counter = 0
            else:
                if iepoch > self.grace:
                    patience_counter += 1
                    if patience_counter >= self.patience:
                        print(f'\nNo progress for {self.patience} epochs. Early stopping at epoch {iepoch}')
                        break

            # Report step value back to Optuna engine for pruning checks
            trial.report(val_rmse, iepoch)

            if trial.should_prune():
                print(f"Trial {trial.number} pruned dynamically by Hyperband at epoch {iepoch}.")
                raise optuna.exceptions.TrialPruned()

        print(f'Trial {trial.number} completed optimization execution. Best score achieved: {best_rmse:.4f} at epoch {best_epoch}')
        return best_rmse

In [ ]:
# Setup Data
input_size = next(iter(train_loader))[0].shape[2]
nepoch = 100

# Early stepping criteria
patience = 20
grace = 20

# Definig the steps of forecast
forecast_steps = 24

# Initialize Objective
objective = Objective(
    train_loader,
    val_loader,
    input_size,
    forecast_steps,
    device,
    nepoch,
    patience,
    grace
)

storage_path = "sqlite:///optuna_studies/pm25_lstm_study.db"
study_name = "pm25_lstm"

# Create Study
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.HyperbandPruner(min_resource=5, max_resource=nepoch),
    sampler=optuna.samplers.TPESampler(seed),
    storage=storage_path,
    study_name=study_name,
    load_if_exists=True
)

n_trials = 50
# Optimize
if len(study.trials) <= n_trials:
    print("Starting optimization...")
    study.optimize(objective, n_trials=n_trials)

/tmp/ipykernel_61753/1704058394.py:31: FutureWarning: __init__() got {'consider_prior'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'consider_prior', 'prior_weight', 'consider_magic_clip', 'consider_endpoints', 'n_startup_trials', 'n_ei_candidates', 'gamma', 'weights', 'seed'] in __init__() have been deprecated since v4.4.0. They will be replaced with the corresponding keyword arguments in v6.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v4.4.0 for details.
  sampler=optuna.samplers.TPESampler(seed),
/tmp/ipykernel_61753/1704058394.py:31: FutureWarning: `consider_prior` has been deprecated in v4.3.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.3.0.
  sampler=optuna.samplers.TPESampler(seed),
[I 2026-06-16 22:31:05,220] Using an existing study with name 'pm25_lstm' instead of creating a new one.


Starting optimization...


Trial 3:  51%|█████     | 51/100 [20:36<19:47, 24.24s/it, Train Loss=7657.4394, Val RMSE=95.6407]
[W 2026-06-16 22:51:41,617] Trial 3 failed with parameters: {'num_layers': 2, 'hidden_size': 16, 'dropout': 0.06777989945058027, 'optim_lr': 0.00012920502706592065, 'l1_lambda': 0.008029243649008939, 'l2_lambda': 0.0010479767161676139} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_61753/3244031673.py", line 81, in __call__
    train_score, train_loss = trainer.train_one_epoch(self.train_loader)
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_61753/3217683423.py", line 35, in train_one_epoch
    y_hat = self.model(X)
            ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.p

KeyboardInterrupt: 